# slide2pptx: Colab setup

Run the cells top to bottom. This clones the repo, installs LibreOffice and fonts, and generates the synthetic dataset into Google Drive so it survives the Colab session.

**Runtime:** data generation only needs CPU. Switch to a T4 GPU runtime later for training.

In [ ]:
# 1. Settings: change REPO_URL to your GitHub repo
REPO_URL = "https://github.com/C00p1r/slide2pptx.git"
N_SLIDES = 5000      # start with 200 to test, then go bigger
DRIVE_DIR = "/content/drive/MyDrive/slide2pptx"

In [ ]:
# 2. Mount Google Drive (dataset is saved there)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p {DRIVE_DIR}

In [ ]:
# 3. Clone the repo (or pull the latest changes)
import os
if not os.path.exists('/content/slide2pptx'):
    !git clone {REPO_URL} /content/slide2pptx
else:
    !git -C /content/slide2pptx pull
%cd /content/slide2pptx

In [ ]:
# 4. Install LibreOffice, poppler and fonts (~2-3 min)
!apt-get -qq update
!apt-get -qq install -y libreoffice-impress poppler-utils fonts-liberation fonts-dejavu fonts-crosextra-carlito fonts-crosextra-caladea fonts-noto-cjk fonts-noto-color-emoji > /dev/null
!apt-get -qq install -y fonts-inter fonts-open-sans fonts-lato > /dev/null 2>&1 || true
!fc-cache -f > /dev/null
!pip install -q -r requirements.txt
from slidegen import fonts
print('Latin fonts:', fonts.latin_fonts())
print('Chinese fonts:', fonts.cjk_fonts())

In [ ]:
# 5. Quick test: 24 slides, then look at the labels
!rm -rf /content/test_synth
!python -m slidegen.generate --out /content/test_synth --n 24 --batch 12 --workers 2
!python -m slidegen.visualize /content/test_synth --n 12
from IPython.display import Image, display
display(Image('/content/test_synth/preview/contact_sheet.png', width=1000))

If the preview looks right, generate the full dataset. Each batch is written as soon as it finishes, so if Colab disconnects you can resume with `--start` set to the number of slides already done.

In [ ]:
# 6. Full dataset into Drive (about 1 slide/s on free Colab: 5000 slides is roughly 1.5 h)
OUT = f"{DRIVE_DIR}/data/synth"
!python -m slidegen.generate --out {OUT} --n {N_SLIDES} --workers 2 --jpg

In [ ]:
# 7. Check counts
!for s in train val test; do echo $s $(ls {OUT}/images/$s | wc -l); done